# 01 — 离散音乐事件提取

**BWV861 音乐形式化分析系统** · Step 1

从 MusicXML 中提取所有离散事件 $x_n$，按声部层 $\lambda=(s,v)$ 进行分层。

实现文档 Section: **离散音列事件**

## 1.1 解析 MusicXML 并提取事件

In [1]:
import sys, os

# 确保项目根目录在 path 中
PROJECT_ROOT = r"e:\大三\Cline Projects\bien_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from music21 import converter
from music_analysis.events import (
    extract_events, build_voice_layers, merge_sparse_voices,
    save_events, load_events, print_event_stats,
)
from music_analysis import config

# 解析 .mxl
print("正在解析 BWV861_Prelude_G_minor.mxl ...")
score = converter.parse("BWV861_Prelude_G_minor.mxl")
print(f"✅ 解析完成: {len(score.parts)} 个声部\n")

# 提取事件
print("正在提取离散音乐事件...")
events = extract_events(score)

# 声部分层
voice_layers = build_voice_layers(events)
print(f"\n原始声部层数: {len(voice_layers)}")

# 合并稀疏声部层 (阈值: MIN_VOICE_EVENTS = {config.MIN_VOICE_EVENTS})
events, voice_layers = merge_sparse_voices(events, voice_layers)

# 统计
print_event_stats(events, voice_layers)

# 持久化
save_events(events, voice_layers)

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.

正在解析 BWV861_Prelude_G_minor.mxl ...
✅ 解析完成: 2 个声部

正在提取离散音乐事件...

原始声部层数: 10
  🔀 稀疏声部合并: 10 → 6 层 (4 个稀疏层共 40 个事件已重分配, 阈值=20)
📊 事件提取统计
  总事件数 N       = 1456
  音符事件         = 1352
  休止符事件       = 104
  含延音线的事件   = 128
  含力度标记的事件 = 0
  含运音法的事件   = 0
  声部层数 |Λ|     = 6
------------------------------------------------------------
  λ=(P1-Staff1,    0): 109 events (106 notes)
  λ=(P1-Staff1,    1): 433 events (404 notes)
  λ=(P1-Staff1,    2): 315 events (288 notes)
  λ=(P1-Staff2,    0): 240 events (232 notes)
  λ=(P1-Staff2,    5): 204 events (182 notes)
  λ=(P1-Staff2,    6): 155 events (140 notes)
✅ 事件数据已保存: data/events.pkl (1456 events)
✅ 声部分层已保存: data/voice_layers.pkl (6 layers)


## 1.2 事件预览

查看前 20 个事件和前 3 个声部层的结构。

In [2]:
import pandas as pd

# 前 20 个事件
df_preview = pd.DataFrame(events[:20])
cols = ["global_index", "p_n", "d_n", "t_n", "s_n", "v_n", "m_n", "r_n", "alpha_n"]
display_cols = [c for c in cols if c in df_preview.columns]
display(df_preview[display_cols])
print(f"\n... 共 {len(events)} 个事件")

# 各声部层预览
print("\n--- 声部层结构预览 ---")
for (s, v), layer_events in sorted(voice_layers.items()):
    n_notes = sum(1 for e in layer_events if e["p_n"] is not None)
    print(f"\nλ = (s='{s}', v='{v}') — 共 {len(layer_events)} 个事件 ({n_notes} notes):")
    for ev in layer_events[:5]:
        tie_info = ev["beta_n"]["tie"] or "-"
        print(f"  idx={ev['global_index']:4d}  p={str(ev['p_n']):>5s}  d={ev['d_n']:5.2f}  t={ev['t_n']:6.1f}  m={ev['m_n']:2d}  r={ev['r_n']:4.1f}  tie={tie_info}")
    if len(layer_events) > 5:
        print(f"  ... ({len(layer_events) - 5} more)")

,global_index,p_n,d_n,t_n,s_n,v_n,m_n,r_n,alpha_n
0,1,79,4.00,0.00,P1-Staff1,0,1,0.00,None
1,2,58,0.25,0.00,P1-Staff2,5,1,0.00,None
2,3,55,0.50,0.00,P1-Staff2,6,1,0.00,None
3,4,62,0.25,0.25,P1-Staff2,5,1,0.25,None
4,5,60,0.25,0.50,P1-Staff2,5,1,0.50,None
5,6,55,0.50,0.50,P1-Staff2,6,1,0.50,None
6,7,63,0.25,0.75,P1-Staff2,5,1,0.75,None
7,8,58,0.25,1.00,P1-Staff2,5,1,1.00,None
8,9,55,0.50,1.00,P1-Staff2,6,1,1.00,None
9,10,62,0.25,1.25,P1-Staff2,5,1,1.25,None



... 共 1456 个事件

--- 声部层结构预览 ---

λ = (s='P1-Staff1', v='0') — 共 109 个事件 (106 notes):
  idx=   1  p=   79  d= 4.00  t=   0.0  m= 1  r= 0.0  tie=start
  idx=  26  p=   79  d= 0.25  t=   4.0  m= 2  r= 0.0  tie=stop
  idx=  29  p=   74  d= 0.25  t=   4.2  m= 2  r= 0.2  tie=-
  idx=  30  p=   75  d= 0.12  t=   4.5  m= 2  r= 0.5  tie=-
  idx=  32  p=   74  d= 0.12  t=   4.6  m= 2  r= 0.6  tie=-
  ... (104 more)

λ = (s='P1-Staff1', v='1') — 共 396 个事件 (370 notes):
  idx= 171  p= None  d= 0.25  t=  24.0  m= 7  r= 0.0  tie=-
  idx= 174  p=   82  d= 0.25  t=  24.2  m= 7  r= 0.2  tie=-
  idx= 175  p=   79  d= 0.25  t=  24.5  m= 7  r= 0.5  tie=-
  idx= 177  p=   82  d= 0.25  t=  24.8  m= 7  r= 0.8  tie=-
  idx= 178  p=   77  d= 0.25  t=  25.0  m= 7  r= 1.0  tie=-
  ... (391 more)

λ = (s='P1-Staff1', v='2') — 共 315 个事件 (288 notes):
  idx= 172  p=   74  d= 0.50  t=  24.0  m= 7  r= 0.0  tie=-
  idx= 176  p=   75  d= 0.50  t=  24.5  m= 7  r= 0.5  tie=-
  idx= 179  p=   74  d= 0.50  t=  25.0  m= 7  r